In [ ]:
!pip install spotipy

In [14]:
import os
import re
import time
import tempfile
import requests
import librosa
import imageio_ffmpeg
import concurrent.futures
import spotipy
import random
import numpy as np
import pandas as pd
from spotipy.oauth2 import SpotifyClientCredentials
from bs4 import BeautifulSoup
from tqdm import tqdm
from dotenv import load_dotenv
from itables import options as opt
from itables import show
from collections import Counter

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

load_dotenv()

# Register FFmpeg for Windows MP3 decoding
try:
    ffmpeg_dir = os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())
    if ffmpeg_dir not in os.environ["PATH"]:
        os.environ["PATH"] += os.pathsep + ffmpeg_dir
except Exception as e:
    print(f"Warning: Could not auto-set FFmpeg path: {e}")

In [16]:
def get_tempo_from_preview(preview_url):
    """Downloads 30s audio preview to a temp file and calculates BPM using librosa."""
    tmp_path = None
    try:
        audio_response = requests.get(preview_url, timeout=10)
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
            tmp_file.write(audio_response.content)
            tmp_path = tmp_file.name
        
        y, sr = librosa.load(tmp_path, sr=None)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        bpm_val = float(np.atleast_1d(tempo)[0])
        return round(bpm_val, 2)
    except Exception as e:
        print(f"\n⚠️ Librosa Error on track: {e}")
        return None
    finally:
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass

def analyze_single_song(song_title, artist_name):
  print(f"🔍 Searching Deezer and Spotify for '{song_title}' by {artist_name}...")

  calculated_bpm = None
  deezer_duration = None
  deezer_genre = "Unknown"
  spotify_url = None

  try:
    global sp
    if "sp" in globals():
      spotify_query = f"track:{song_title} artist:{artist_name}"
      spotify_res = sp.search(q=spotify_query, type="track", limit=1)
      tracks_found = spotify_res.get("tracks", {}).get("items", [])
      if tracks_found:
        track_id = tracks_found[0]["id"]
        spotify_url = f"https://open.spotify.com/track/{track_id}"
  except Exception as e:
    print(f"⚠️ Could not fetch Spotify URL automatically: {e}")

  deezer_query = f"{song_title} {artist_name}"
  search_url = (
      f"https://api.deezer.com/search?q={requests.utils.quote(deezer_query)}"
  )

  try:
    deezer_res = requests.get(search_url).json()

    if deezer_res.get("data"):
      track_data = deezer_res["data"][0]
      preview_url = track_data.get("preview")
      deezer_duration = track_data.get("duration")
      album_id = track_data.get("album", {}).get("id")

      if preview_url:
        calculated_bpm = get_tempo_from_preview(preview_url)

      if album_id:
        album_res = requests.get(
            f"https://api.deezer.com/album/{album_id}"
        ).json()
        genres_data = album_res.get("genres", {}).get("data", [])
        if genres_data:
          deezer_genre = genres_data[0].get("name", "Unknown")

      print("\n✅ Match Found!")
      print("-" * 20)
      print(f"Track:    {track_data.get('title')}")
      print(f"Artist:   {track_data.get('artist', {}).get('name')}")
      print(f"Genre:    {deezer_genre}")
      print(f"Duration: {deezer_duration} seconds")

      if calculated_bpm:
        print(f"BPM:      {calculated_bpm:.1f}")
      else:
        print("BPM:      ⚠️ No audio preview available to calculate.")

      # Automatically trigger the stream counter if a Spotify URL was found
      if spotify_url:
        get_exact_stream_count(spotify_url)
      else:
        print("Streams:  ⚠️ Spotify URL could not be resolved automatically.")

      print("-" * 20)

    else:
      print(f"\n❌ Could not find any matches for '{song_title}' by {artist_name}.")

  except Exception as e:
    print(f"\n❌ API Error: {e}")

def get_exact_stream_count(track_url):
    print(f"Launching headless browser for: {track_url}")
    
    #Setup Chrome Options (Headless mode so it runs invisibly)
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    
    # Disguise the automated bot as a normal human browser to prevent blocking
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    #Initialize the Web Driver automatically
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    try:
        driver.get(track_url)
        
        #Wait for the play count element to render. 
        #Spotify uses a specific 'data-testid' attribute for their play count on the web player.
        element = WebDriverWait(driver, 12).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-testid='playcount']"))
        )
        
        stream_count = element.text
        print(f"\n✅ Success! Exact Stream Count: {stream_count}")
        return stream_count

    except Exception as e:
        print("\n❌ Could not retrieve stream count.")
        print("Spotify might have changed their HTML class names, or they are serving a captcha.")
        print(f"Error details: {e}")
        return None
        
    finally:
        # Always quit the driver to prevent invisible Chrome windows from eating your RAM
        driver.quit()

In [14]:
analyze_single_song("Baby Steps", "Quadeca")
analyze_single_song("babygirl", "JPEGMAFIA")

🔍 Searching Deezer and Spotify for 'Baby Steps' by Quadeca...

✅ Match Found!
--------------------
Track:    Baby Steps
Artist:   Quadeca
Genre:    Alternative
Duration: 239 seconds
BPM:      90.7
🕵️‍♂️ Launching headless browser for: https://open.spotify.com/track/40eANvl45mhgwEHL5pHHSn

✅ Success! Exact Stream Count: 1,018,820
--------------------
🔍 Searching Deezer and Spotify for 'babygirl' by JPEGMAFIA...

✅ Match Found!
--------------------
Track:    babygirl
Artist:   JPEGMAFIA
Genre:    Rap/Hip Hop
Duration: 147 seconds
BPM:      139.7
🕵️‍♂️ Launching headless browser for: https://open.spotify.com/track/4aayS1uReTPeEVCOVTPm6n

✅ Success! Exact Stream Count: 3,111,670
--------------------


In [15]:
SPOTIPY_CLIENT_ID = os.getenv("SPOTIFY_ID")
SPOTIPY_CLIENT_SECRET = os.getenv("SPOTIFY_SECRET_ID")
if not SPOTIPY_CLIENT_ID or not SPOTIPY_CLIENT_SECRET:
    print("❌ Error: Missing Spotify credentials. Check your .env file!")
else:
    sp = spotipy.Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=SPOTIPY_CLIENT_ID, 
            client_secret=SPOTIPY_CLIENT_SECRET
        )
    )

    # Playlist created of the new music friday
    playlist_id = "1mmrHLMHjYGEHqDN11QibR"
    print("Fetching the latest New Music Friday playlist from Spotify...")

    try:
        results = sp.playlist_tracks(playlist_id)
        tracks = results["items"]

        random_selection = random.sample(tracks, 2)
        print(f"2 songs grabbed...Starting analysis...\n")
        print("=" * 40)

        for item in random_selection:
            track = item["track"]
            
            song_title = track["name"]
            artist_name = track["artists"][0]["name"]
            
            analyze_single_song(song_title, artist_name)
            
            time.sleep(3)

    except Exception as e:
        print(f"❌ Spotify API Error: {e}")

Fetching the latest New Music Friday playlist from Spotify...
2 songs grabbed...Starting analysis...

🔍 Searching Deezer and Spotify for 'ZIZI' by Ozuna...


C:\Users\xboxc\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✅ Match Found!
--------------------
Track:    ZIZI
Artist:   Ozuna
Genre:    Rap/Hip Hop
Duration: 243 seconds
BPM:      172.3
🕵️‍♂️ Launching headless browser for: https://open.spotify.com/track/5BsvzSvw98mLqpZznMjuLX

✅ Success! Exact Stream Count: 994,468
--------------------
🔍 Searching Deezer and Spotify for 'In & Out' by Ravyn Lenae...

✅ Match Found!
--------------------
Track:    In & Out
Artist:   Ravyn Lenae
Genre:    R&B
Duration: 200 seconds
BPM:      136.0
🕵️‍♂️ Launching headless browser for: https://open.spotify.com/track/6KLLoGHAIYPfNYfRo4U1Ql

✅ Success! Exact Stream Count: 14,271
--------------------
